In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Load dataset
df = pd.read_csv("custom_sample_mass_radius_k2_5.csv")
X = df.drop(columns=["Type"]).values.astype(np.float32)
y = df["Type"].values

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, train_size=0.75, random_state=42)





In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from skorch import NeuralNetClassifier

class FlexibleModel(nn.Module):
    def __init__(self, in_features, hidden_layers, out_features, dropout_p=0.3):
        """
        in_features: int — input features
        hidden_layers: list of ints — neurons in each hidden layer
        out_features: int — number of outputs (classes)
        dropout_p: float — dropout probability
        """
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden_layers:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            # layers.append(nn.Dropout(p=dropout_p))
            prev = h
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(prev, out_features)

    def forward(self, x):
        x = self.hidden(x)
        x = self.out(x)
        return x


X_train_ = X_train.astype(np.float32)
X_test_ = X_test.astype(np.float32)
y_train_ = y_train.astype(np.longlong)
y_test_ = y_test.astype(np.longlong)

input_dim = X_train_.shape[1]
output_dim = len(np.unique(y_train_))  


net = NeuralNetClassifier(
    module=FlexibleModel,
    module__in_features=input_dim,
    module__hidden_layers=[32, 64, 32],
    module__out_features=output_dim,
    max_epochs=20,                
    lr=0.001,                     
    optimizer=torch.optim.Adam,
    criterion=nn.CrossEntropyLoss,
    batch_size=64,
    verbose=0,
)

param_grid = {
    'module__hidden_layers': [
        [64, 64],
        [32, 64, 32],
        [128, 64], [128, 64, 32],
        [256, 128, 64],
        [254, 254, 254, 254]],
    'lr': [0.01, 0.001, 0.0005],
    'batch_size': [16, 32, 64, 128, 256],
    'max_epochs': [100, 500, 1000]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs = GridSearchCV(net, param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=2)
gs.fit(X_train_, y_train_)

print("\n Best parameters found:\n", gs.best_params_)
print(" Best cross-val accuracy: ", gs.best_score_)

best_net = gs.best_estimator_
y_pred = best_net.predict(X_test_)

print("\n Test accuracy: ", accuracy_score(y_test_, y_pred))
print("\nClassification report:\n", classification_report(y_test_, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test_, y_pred))


In [ ]:
# ==========================================================
# Train and Save Neural Network (PyTorch) — Safe Scaler Handling
# ==========================================================

import os
import numpy as np
import torch
import torch.nn as nn
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

class FlexibleModel(nn.Module):
    def __init__(self, in_features, hidden_layers, out_features, dropout_p=0.3):
        """
        in_features: int — input features
        hidden_layers: list of ints — neurons in each hidden layer
        out_features: int — number of outputs (classes)
        dropout_p: float — dropout probability
        """
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden_layers:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            # layers.append(nn.Dropout(p=dropout_p))
            prev = h
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(prev, out_features)

    def forward(self, x):
        x = self.hidden(x)
        x = self.out(x)
        return x

SAVE_DIR = "pretrained_models"  # Change to 'pretrained_models' if needed
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_NAME = "Neural_Network"
MODEL_PATH = os.path.join(SAVE_DIR, f"{MODEL_NAME}.pth")
ARCH_PATH = os.path.join(SAVE_DIR, f"{MODEL_NAME}_arch.joblib")
SCALER_PATH = os.path.join(SAVE_DIR, "scaler.joblib")
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Correct fitted check for MinMaxScaler
if hasattr(scaler, "data_min_") and hasattr(scaler, "data_max_"):
    joblib.dump(scaler, SCALER_PATH)
    print(f"💾 MinMaxScaler saved to: {SCALER_PATH}")
else:
    raise RuntimeError("❌ MinMaxScaler not fitted — check your data.")
# ----------------------------------------------------------
# Data loaders
# ----------------------------------------------------------
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# ----------------------------------------------------------
# Model setup
# ----------------------------------------------------------
hidden_layers = [32, 64]
model = FlexibleModel(in_features=X_train.shape[1], hidden_layers=hidden_layers, out_features=1)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
max_epochs = 200

# ----------------------------------------------------------
# Training loop
# ----------------------------------------------------------
epoch_losses = []

for epoch in range(max_epochs):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    epoch_losses.append(avg_loss)

    print(f"Epoch [{epoch+1}/{max_epochs}] | Loss: {avg_loss:.6f}")


# ----------------------------------------------------------
# Evaluation
# ----------------------------------------------------------
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor)
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).long().view(-1)

acc = accuracy_score(y_test_tensor.numpy(), preds.numpy())
print(f"\n✅ Test Accuracy: {acc*100:.2f}%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test_tensor.numpy(), preds.numpy()))
print("\nClassification Report:\n", classification_report(y_test_tensor.numpy(), preds.numpy()))

# ----------------------------------------------------------
# Save model + architecture safely
# ----------------------------------------------------------
torch.save(model.state_dict(), MODEL_PATH)
joblib.dump(
    {"in_features": X_train.shape[1], "hidden_layers": hidden_layers, "out_features": 1},
    ARCH_PATH
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.figure(figsize=(7, 5))
epochs = np.arange(1, max_epochs + 1)
plt.figure(figsize=(7, 5))
plt.semilogy(
    epochs,
    epoch_losses,
    lw=2.5,
    color="#1f77b4",
    label="Training Loss"
)
# Optional smoothing
window = 5
if len(epoch_losses) >= window:
    smoothed = np.convolve(epoch_losses, np.ones(window)/window, mode="valid")
    plt.semilogy(
        epochs[window-1:],
        smoothed,
        lw=2,
        linestyle="--",
        color="#ff7f0e",
        label="Smoothed Loss"
    )
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Neural Network Training Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()